<a href="https://colab.research.google.com/github/YASH-HASH-PIXEL/-ANN-Linear_Regression/blob/main/MCP_SERVER_PROJECT_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Self-contained MCP server implementation for Jupyter
import asyncio
import json
from datetime import datetime

class SimpleMCPServer:
    """Minimal MCP-like server that works in Jupyter"""

    def __init__(self, name):
        self.name = name
        self.tools = {}

    def register_tool(self, name, handler, description, input_schema=None):
        """Register a tool with the server"""
        self.tools[name] = {
            "handler": handler,
            "description": description,
            "input_schema": input_schema or {}
        }

    async def list_tools(self):
        """List all registered tools"""
        tools_list = []
        for name, info in self.tools.items():
            tools_list.append({
                "name": name,
                "description": info["description"],
                "inputSchema": info["input_schema"]
            })
        return tools_list

    async def call_tool(self, name, arguments):
        """Call a registered tool"""
        if name not in self.tools:
            return {"error": f"Tool '{name}' not found"}

        try:
            result = await self.tools[name]["handler"](arguments)
            return result
        except Exception as e:
            return {"error": str(e)}

# Create and configure server
server = SimpleMCPServer("Jupyter-MCP")

# Register tools
async def handle_greet(args):
    name = args.get("name", "World")
    return {
        "content": [{
            "type": "text",
            "text": f" Hello {name}! This is MCP running in Jupyter!"
        }]
    }

async def handle_current_time(args):
    now = datetime.now()
    fmt = args.get("format", "full")
    if fmt == "time":
        text = now.strftime("%H:%M:%S")
    elif fmt == "date":
        text = now.strftime("%Y-%m-%d")
    else:
        text = now.strftime("%Y-%m-%d %H:%M:%S")
    return {
        "content": [{
            "type": "text",
            "text": f" Current {fmt}: {text}"
        }]
    }

async def handle_calculator(args):
    op = args.get("operation", "add")
    a = float(args["a"])
    b = float(args["b"])

    ops = {
        "add": ("+", a + b),
        "subtract": ("-", a - b),
        "multiply": ("×", a * b),
        "divide": ("÷", a / b if b != 0 else "Error: division by zero")
    }

    symbol, result = ops[op]
    return {
        "content": [{
            "type": "text",
            "text": f" {a} {symbol} {b} = {result}"
        }]
    }

async def handle_data_analysis(args):
    numbers = args.get("numbers", [])
    if not numbers:
        return {"content": [{"type": "text", "text": "No data provided"}]}

    avg = sum(numbers) / len(numbers)
    return {
        "content": [{
            "type": "text",
            "text": f""" Data Analysis:
  Count: {len(numbers)}
  Sum: {sum(numbers)}
  Average: {avg:.2f}
  Max: {max(numbers)}
  Min: {min(numbers)}"""
        }]
    }

# Register all tools
server.register_tool("greet", handle_greet, "Greet someone", {
    "type": "object",
    "properties": {"name": {"type": "string"}},
    "required": ["name"]
})

server.register_tool("get_time", handle_current_time, "Get current time", {
    "type": "object",
    "properties": {"format": {"type": "string", "enum": ["full", "time", "date"]}}
})

server.register_tool("calculate", handle_calculator, "Perform calculations", {
    "type": "object",
    "properties": {
        "operation": {"type": "string", "enum": ["add", "subtract", "multiply", "divide"]},
        "a": {"type": "number"},
        "b": {"type": "number"}
    },
    "required": ["operation", "a", "b"]
})

server.register_tool("analyze", handle_data_analysis, "Analyze number list", {
    "type": "object",
    "properties": {"numbers": {"type": "array", "items": {"type": "number"}}},
    "required": ["numbers"]
})

# Test the server

print(" Testing MCP Server in Jupyter")


# List tools
tools = await server.list_tools()
print(f"\n Available Tools ({len(tools)}):")
for tool in tools:
    print(f"   {tool['name']}: {tool['description']}")

# Test greet

result = await server.call_tool("greet", {"name": "Data Scientist"})
print(result["content"][0]["text"])

# Test time
result = await server.call_tool("get_time", {"format": "full"})
print(result["content"][0]["text"])

# Test calculator
result = await server.call_tool("calculate", {
    "operation": "divide",
    "a": 25,
    "b": 5
})
print(result["content"][0]["text"])

# Test data analysis
result = await server.call_tool("analyze", {
    "numbers": [23, 45, 12, 67, 34, 89, 21, 56]
})
print(result["content"][0]["text"])

 Testing MCP Server in Jupyter

 Available Tools (4):
   greet: Greet someone
   get_time: Get current time
   calculate: Perform calculations
   analyze: Analyze number list
 Hello Data Scientist! This is MCP running in Jupyter!
 Current full: 2026-07-11 08:20:15
 25.0 ÷ 5.0 = 5.0
 Data Analysis:
  Count: 8
  Sum: 347
  Average: 43.38
  Max: 89
  Min: 12


In [ ]:
pip install mcp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.6 MB/s eta 0:00:00


In [ ]:
# Self-contained MCP implementation for Jupyter testing
import json
import os
from datetime import datetime
from typing import Dict, List, Any

class MCPServer:
    """MCP-like server that works in Jupyter"""

    def __init__(self, name: str):
        self.name = name
        self.tools = {}

    def tool(self, name: str, description: str, input_schema: Dict = None):
        """Decorator to register a tool"""
        def decorator(func):
            self.tools[name] = {
                "handler": func,
                "description": description,
                "input_schema": input_schema or {
                    "type": "object",
                    "properties": {}
                }
            }
            return func
        return decorator

    def get_tools(self) -> List[Dict]:
        """List all registered tools"""
        tools_list = []
        for name, info in self.tools.items():
            tools_list.append({
                "name": name,
                "description": info["description"],
                "inputSchema": info["input_schema"]
            })
        return tools_list

    async def call_tool(self, name: str, arguments: Dict = None) -> Dict:
        """Call a registered tool"""
        if name not in self.tools:
            return {"error": f"Tool '{name}' not found"}

        try:
            if arguments:
                result = await self.tools[name]["handler"](**arguments)
            else:
                result = await self.tools[name]["handler"]()
            return {"success": True, "result": result}
        except Exception as e:
            return {"error": str(e)}

# Create and configure the server
server = MCPServer("notes-keeper")
NOTES_FILE = "jupyter_notes.json"

def load_notes():
    if os.path.exists(NOTES_FILE):
        with open(NOTES_FILE, 'r') as f:
            return json.load(f)
    return []

def save_notes(notes):
    with open(NOTES_FILE, 'w') as f:
        json.dump(notes, f, indent=2)

# Register tools using decorators
@server.tool(
    name="add_note",
    description="Save a quick note with optional tags",
    input_schema={
        "type": "object",
        "properties": {
            "title": {"type": "string", "description": "Title of the note"},
            "content": {"type": "string", "description": "Content of the note"},
            "tags": {"type": "string", "description": "Comma-separated tags"}
        },
        "required": ["title", "content"]
    }
)
async def add_note(title: str, content: str, tags: str = ""):
    """Save a quick note"""
    notes = load_notes()
    note = {
        "id": len(notes) + 1,
        "title": title,
        "content": content,
        "tags": [t.strip() for t in tags.split(",") if t.strip()],
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    notes.append(note)
    save_notes(notes)
    return f" Note #{note['id']} saved: {title}"

@server.tool(
    name="list_notes",
    description="Show all notes or filter by tag",
    input_schema={
        "type": "object",
        "properties": {
            "filter_tag": {"type": "string", "description": "Filter by tag"}
        }
    }
)
async def list_notes(filter_tag: str = ""):
    """Show all saved notes"""
    notes = load_notes()

    if filter_tag:
        notes = [n for n in notes if filter_tag in n.get("tags", [])]

    if not notes:
        return " No notes found!"

    result = " Your Notes:\n" + "=" * 30 + "\n"
    for note in notes:
        tags_str = f" [ {', '.join(note['tags'])}]" if note.get('tags') else ""
        result += f"#{note['id']} {note['title']}{tags_str}\n"

    return result

@server.tool(
    name="read_note",
    description="Read a specific note by ID",
    input_schema={
        "type": "object",
        "properties": {
            "note_id": {"type": "integer", "description": "ID of note to read"}
        },
        "required": ["note_id"]
    }
)
async def read_note(note_id: int):
    """Read a specific note"""
    notes = load_notes()

    for note in notes:
        if note["id"] == note_id:
            result = f" Note #{note['id']}: {note['title']}\n"
            result += "=" * 40 + "\n\n"
            result += note['content'] + "\n\n"
            result += f" Created: {note.get('created_at', 'Unknown')}"
            if note.get('tags'):
                result += f"\n Tags: {', '.join(note['tags'])}"
            return result

    return f" Note #{note_id} not found"

@server.tool(
    name="search_notes",
    description="Search notes by keyword",
    input_schema={
        "type": "object",
        "properties": {
            "keyword": {"type": "string", "description": "Keyword to search"}
        },
        "required": ["keyword"]
    }
)
async def search_notes(keyword: str):
    """Search notes by keyword"""
    notes = load_notes()
    keyword = keyword.lower()

    matches = [
        n for n in notes
        if keyword in n['title'].lower() or keyword in n['content'].lower()
    ]

    if not matches:
        return f" No notes found containing '{keyword}'"

    result = f" Found {len(matches)} notes:\n" + "=" * 40 + "\n"
    for note in matches:
        result += f"#{note['id']}: {note['title']}\n"
        content = note['content']
        if len(content) > 100:
            content = content[:100] + "..."
        result += f"   {content}\n\n"

    return result

# Test the server in Jupyter

print(" Testing MCP Server in Jupyter")


# List tools
print("\n Available Tools:")
for tool in server.get_tools():
    print(f"   {tool['name']}: {tool['description']}")

# Add notes

print("1️ Adding notes...")
result = await server.call_tool("add_note", {
    "title": "Shopping List",
    "content": "Buy milk, eggs, bread, and butter",
    "tags": "shopping,personal"
})
print(result["result"])

result = await server.call_tool("add_note", {
    "title": "Meeting Notes",
    "content": "Discuss Q4 targets and budget review",
    "tags": "work,meeting"
})
print(result["result"])

result = await server.call_tool("add_note", {
    "title": "Project Ideas",
    "content": "Build MCP server, Create web app, Learn AI tools",
    "tags": "work,ideas"
})
print(result["result"])

# List all notes

print("2️ Listing all notes...")
result = await server.call_tool("list_notes")
print(result["result"])

# Filter by tag

print("3️ Filtering by 'work' tag...")
result = await server.call_tool("list_notes", {"filter_tag": "work"})
print(result["result"])

# Read specific note

print("4️ Reading note #2...")
result = await server.call_tool("read_note", {"note_id": 2})
print(result["result"])

# Search notes

print("5️ Searching for 'AI'...")
result = await server.call_tool("search_notes", {"keyword": "AI"})
print(result["result"])


print(" All tests completed successfully!")

 Testing MCP Server in Jupyter

 Available Tools:
   add_note: Save a quick note with optional tags
   list_notes: Show all notes or filter by tag
   read_note: Read a specific note by ID
   search_notes: Search notes by keyword
1️ Adding notes...
 Note #1 saved: Shopping List
 Note #2 saved: Meeting Notes
 Note #3 saved: Project Ideas
2️ Listing all notes...
 Your Notes:
#1 Shopping List [ shopping, personal]
#2 Meeting Notes [ work, meeting]
#3 Project Ideas [ work, ideas]

3️ Filtering by 'work' tag...
 Your Notes:
#2 Meeting Notes [ work, meeting]
#3 Project Ideas [ work, ideas]

4️ Reading note #2...
 Note #2: Meeting Notes

Discuss Q4 targets and budget review

 Created: 2026-07-11 08:20:22
 Tags: work, meeting
5️ Searching for 'AI'...
 Found 1 notes:
#3: Project Ideas
   Build MCP server, Create web app, Learn AI tools


 All tests completed successfully!


In [ ]:
import json
import os
from datetime import datetime, timedelta
from typing import Dict, List, Any

class MCPServer:
    """MCP-like server that works in Jupyter"""

    def __init__(self, name: str):
        self.name = name
        self.tools = {}

    def tool(self, name: str, description: str, input_schema: Dict = None):
        """Decorator to register a tool"""
        def decorator(func):
            self.tools[name] = {
                "handler": func,
                "description": description,
                "input_schema": input_schema or {"type": "object", "properties": {}}
            }
            return func
        return decorator

    def get_tools(self) -> List[Dict]:
        """List all registered tools"""
        return [
            {"name": name, "description": info["description"], "inputSchema": info["input_schema"]}
            for name, info in self.tools.items()
        ]

    async def call_tool(self, name: str, arguments: Dict = None) -> Dict:
        """Call a registered tool"""
        if name not in self.tools:
            return {"error": f"Tool '{name}' not found"}

        try:
            if arguments:
                result = await self.tools[name]["handler"](**arguments)
            else:
                result = await self.tools[name]["handler"]()
            return {"success": True, "result": result}
        except Exception as e:
            return {"error": str(e)}

# Create enhanced server
server = MCPServer("smart-notes")
NOTES_FILE = "smart_notes.json"

def load_notes():
    if os.path.exists(NOTES_FILE):
        with open(NOTES_FILE, 'r') as f:
            return json.load(f)
    return []

def save_notes(notes):
    with open(NOTES_FILE, 'w') as f:
        json.dump(notes, f, indent=2)

# 1. Basic CRUD Operations
@server.tool(
    name="add_note",
    description="Create a new note with title and content",
    input_schema={
        "type": "object",
        "properties": {
            "title": {"type": "string", "description": "Note title"},
            "content": {"type": "string", "description": "Note content"},
            "priority": {"type": "string", "enum": ["high", "medium", "low"], "description": "Priority level"},
            "category": {"type": "string", "description": "Category like work, personal, ideas"},
            "tags": {"type": "string", "description": "Comma-separated tags"}
        },
        "required": ["title", "content"]
    }
)
async def add_note(title: str, content: str, priority: str = "medium", category: str = "general", tags: str = ""):
    """Add a new note"""
    notes = load_notes()

    note = {
        "id": len(notes) + 1,
        "title": title,
        "content": content,
        "priority": priority,
        "category": category,
        "tags": [t.strip() for t in tags.split(",") if t.strip()],
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "updated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "completed": False
    }
    notes.append(note)
    save_notes(notes)

    priority_emoji = {"high": "🔴", "medium": "🟡", "low": "🟢"}
    return f"{priority_emoji.get(priority, '')} Note #{note['id']} created: {title}"

# 2. Smart Listing
@server.tool(
    name="list_notes",
    description="List notes with filtering and sorting options",
    input_schema={
        "type": "object",
        "properties": {
            "category": {"type": "string", "description": "Filter by category"},
            "priority": {"type": "string", "description": "Filter by priority (high/medium/low)"},
            "tag": {"type": "string", "description": "Filter by tag"},
            "status": {"type": "string", "enum": ["all", "active", "completed"], "description": "Filter by status"},
            "sort_by": {"type": "string", "enum": ["date", "priority", "title"], "description": "Sort order"}
        }
    }
)
async def list_notes(category: str = "", priority: str = "", tag: str = "",
                     status: str = "all", sort_by: str = "date"):
    """List notes with filters"""
    notes = load_notes()

    # Apply filters
    if category:
        notes = [n for n in notes if n.get("category") == category]
    if priority:
        notes = [n for n in notes if n.get("priority") == priority]
    if tag:
        notes = [n for n in notes if tag in n.get("tags", [])]
    if status == "active":
        notes = [n for n in notes if not n.get("completed", False)]
    elif status == "completed":
        notes = [n for n in notes if n.get("completed", False)]

    # Apply sorting
    if sort_by == "priority":
        priority_order = {"high": 0, "medium": 1, "low": 2}
        notes.sort(key=lambda n: priority_order.get(n.get("priority"), 1))
    elif sort_by == "title":
        notes.sort(key=lambda n: n["title"].lower())
    else:  # date
        notes.sort(key=lambda n: n["created_at"], reverse=True)

    if not notes:
        return " No notes found matching your criteria!"

    # Format output
    result = f" Notes ({len(notes)} found)"

    for note in notes:
        priority_emoji = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(note.get("priority"), "")
        status_mark = "✅" if note.get("completed") else "📌"
        category_tag = f"[{note.get('category', 'general')}]"

        result += f"{status_mark} {priority_emoji} #{note['id']} {note['title']} {category_tag}\n"

        # Show tags if any
        if note.get("tags"):
            result += f"    {', '.join(note['tags'])}\n"

    return result

# 3. Smart Search
@server.tool(
    name="search_notes",
    description="Full-text search across all notes",
    input_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search query"},
            "search_in": {"type": "string", "enum": ["all", "title", "content", "tags"], "description": "Where to search"}
        },
        "required": ["query"]
    }
)
async def search_notes(query: str, search_in: str = "all"):
    """Search notes intelligently"""
    notes = load_notes()
    query = query.lower()

    matches = []
    for note in notes:
        found = False
        context = ""

        if search_in in ["all", "title"] and query in note["title"].lower():
            found = True
            context = f"Title match"
        elif search_in in ["all", "content"] and query in note["content"].lower():
            found = True
            # Extract context
            content_lower = note["content"].lower()
            idx = content_lower.find(query)
            start = max(0, idx - 30)
            end = min(len(note["content"]), idx + len(query) + 30)
            snippet = note["content"][start:end]
            if start > 0:
                snippet = "..." + snippet
            if end < len(note["content"]):
                snippet = snippet + "..."
            context = snippet
        elif search_in in ["all", "tags"] and any(query in tag.lower() for tag in note.get("tags", [])):
            found = True
            context = f"Tag match: {', '.join(note['tags'])}"

        if found:
            matches.append({"note": note, "context": context})

    if not matches:
        return f" No notes found matching '{query}'"

    result = f" Found {len(matches)} notes matching '{query}'\n{'='*50}\n"
    for match in matches:
        note = match["note"]
        result += f"📄 #{note['id']}: {note['title']}\n"
        result += f"   {match['context']}\n"
        result += f"   📅 {note['created_at'][:10]}\n\n"

    return result

# 4. Update Notes
@server.tool(
    name="update_note",
    description="Update an existing note",
    input_schema={
        "type": "object",
        "properties": {
            "note_id": {"type": "integer", "description": "ID of note to update"},
            "title": {"type": "string", "description": "New title (optional)"},
            "content": {"type": "string", "description": "New content (optional)"},
            "priority": {"type": "string", "enum": ["high", "medium", "low"]},
            "completed": {"type": "boolean", "description": "Mark as completed"}
        },
        "required": ["note_id"]
    }
)
async def update_note(note_id: int, title: str = None, content: str = None,
                      priority: str = None, completed: bool = None):
    """Update a note"""
    notes = load_notes()

    for note in notes:
        if note["id"] == note_id:
            if title:
                note["title"] = title
            if content:
                note["content"] = content
            if priority:
                note["priority"] = priority
            if completed is not None:
                note["completed"] = completed
            note["updated_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            save_notes(notes)
            return f" Note #{note_id} updated successfully!"

    return f" Note #{note_id} not found"

# 5. Statistics & Insights
@server.tool(
    name="get_stats",
    description="Get statistics about your notes",
    input_schema={"type": "object", "properties": {}}
)
async def get_stats():
    """Get note statistics"""
    notes = load_notes()

    if not notes:
        return " No notes to analyze!"

    total = len(notes)
    completed = len([n for n in notes if n.get("completed")])
    active = total - completed

    # Category breakdown
    categories = {}
    for note in notes:
        cat = note.get("category", "general")
        categories[cat] = categories.get(cat, 0) + 1

    # Priority breakdown
    priorities = {}
    for note in notes:
        pri = note.get("priority", "medium")
        priorities[pri] = priorities.get(pri, 0) + 1

    # Most used tags
    tags_count = {}
    for note in notes:
        for tag in note.get("tags", []):
            tags_count[tag] = tags_count.get(tag, 0) + 1

    top_tags = sorted(tags_count.items(), key=lambda x: x[1], reverse=True)[:5]

    result = f""" Notes Statistics
{'='*50}
 Total Notes: {total}
 Completed: {completed} ({completed*100//total if total > 0 else 0}%)
 Active: {active} ({active*100//total if total > 0 else 0}%)

 Categories:
"""
    for cat, count in sorted(categories.items()):
        bar = "█" * (count * 20 // total if total > 0 else 0)
        result += f"  {cat:15} {count:3} {bar}\n"

    result += "\n Priorities:\n"
    for pri in ["high", "medium", "low"]:
        count = priorities.get(pri, 0)
        result += f"  {pri:8} {count:3} notes\n"

    if top_tags:
        result += f"\n Top Tags:\n"
        for tag, count in top_tags:
            result += f"  {tag}: {count}\n"

    return result

# Test the enhanced server

print(" Enhanced MCP Notes Server")


# Show all tools
print("\n📋 Available Tools:")
for tool in server.get_tools():
    print(f" {tool['name']}: {tool['description']}")

# Simulate real usage

print(" Simulating AI Assistant Usage")


# User: "Add a high priority work task"
print("\n User: Add a high priority work task about Q4 review")
result = await server.call_tool("add_note", {
    "title": "Prepare Q4 Review",
    "content": "Create presentation for Q4 review meeting. Include revenue stats, growth metrics, and next year projections.",
    "priority": "high",
    "category": "work",
    "tags": "meeting,presentation,Q4,important"
})
print(f" Assistant: {result['result']}")

# User: "Add shopping list"
print("\n User: Remind me to buy groceries")
result = await server.call_tool("add_note", {
    "title": "Weekly Groceries",
    "content": "Milk, eggs, bread, vegetables, fruits, chicken, rice",
    "priority": "medium",
    "category": "personal",
    "tags": "shopping,groceries"
})
print(f" Assistant: {result['result']}")

# User: "Show my work tasks"
print("\n User: Show my work notes sorted by priority")
result = await server.call_tool("list_notes", {
    "category": "work",
    "sort_by": "priority"
})
print(f" Assistant:\n{result['result']}")

# User: "Search for meeting notes"
print(" User: Find all notes about meetings")
result = await server.call_tool("search_notes", {
    "query": "meeting"
})
print(f" Assistant:\n{result['result']}")

# User: "Mark Q4 review as done"
print(" User: Mark the Q4 review as completed")
result = await server.call_tool("update_note", {
    "note_id": 10,  # Latest note ID
    "completed": True
})
print(f"Assistant: {result['result']}")

# User: "Show statistics"
print("\n User: Give me a summary of all my notes")
result = await server.call_tool("get_stats", {})
print(f" Assistant:\n{result['result']}")


print(" Server ready for AI integration!")

 Enhanced MCP Notes Server

📋 Available Tools:
 add_note: Create a new note with title and content
 list_notes: List notes with filtering and sorting options
 search_notes: Full-text search across all notes
 update_note: Update an existing note
 get_stats: Get statistics about your notes
 Simulating AI Assistant Usage

 User: Add a high priority work task about Q4 review
 Assistant: 🔴 Note #1 created: Prepare Q4 Review

 User: Remind me to buy groceries
 Assistant: 🟡 Note #2 created: Weekly Groceries

 User: Show my work notes sorted by priority
 Assistant:
 Notes (1 found)📌 🔴 #1 Prepare Q4 Review [work]
    meeting, presentation, Q4, important

 User: Find all notes about meetings
 Assistant:
 Found 1 notes matching 'meeting'
📄 #1: Prepare Q4 Review
   ...te presentation for Q4 review meeting. Include revenue stats, growt...
   📅 2026-07-11


 User: Mark the Q4 review as completed
Assistant:  Note #10 not found

 User: Give me a summary of all my notes
 Assistant:
 Notes Statistics
 T

In [ ]:
# Enhanced MCP Notes Server - Fixed Version
import json
import os
from datetime import datetime
from typing import Dict, List, Any

class MCPServer:
    """MCP-like server that works in Jupyter"""

    def __init__(self, name: str):
        self.name = name
        self.tools = {}

    def tool(self, name: str, description: str, input_schema: Dict = None):
        """Decorator to register a tool"""
        def decorator(func):
            self.tools[name] = {
                "handler": func,
                "description": description,
                "input_schema": input_schema or {"type": "object", "properties": {}}
            }
            return func
        return decorator

    def get_tools(self) -> List[Dict]:
        """List all registered tools"""
        return [
            {"name": name, "description": info["description"], "inputSchema": info["input_schema"]}
            for name, info in self.tools.items()
        ]

    async def call_tool(self, name: str, arguments: Dict = None) -> Dict:
        """Call a registered tool"""
        if name not in self.tools:
            return {"error": f"Tool '{name}' not found"}

        try:
            if arguments:
                result = await self.tools[name]["handler"](**arguments)
            else:
                result = await self.tools[name]["handler"]()
            return {"success": True, "result": result}
        except Exception as e:
            return {"error": str(e)}

# Create enhanced server
server = MCPServer("smart-notes")
NOTES_FILE = "smart_notes.json"

def load_notes():
    if os.path.exists(NOTES_FILE):
        with open(NOTES_FILE, 'r') as f:
            return json.load(f)
    return []

def save_notes(notes):
    with open(NOTES_FILE, 'w') as f:
        json.dump(notes, f, indent=2)

# 1. Add Note with Priority & Category
@server.tool(
    name="add_note",
    description="Create a new note with title, content, priority, and category",
    input_schema={
        "type": "object",
        "properties": {
            "title": {"type": "string", "description": "Note title"},
            "content": {"type": "string", "description": "Note content"},
            "priority": {
                "type": "string",
                "enum": ["high", "medium", "low"],
                "description": "Priority level"
            },
            "category": {
                "type": "string",
                "description": "Category (work, personal, ideas, etc.)"
            },
            "tags": {
                "type": "string",
                "description": "Comma-separated tags"
            }
        },
        "required": ["title", "content"]
    }
)
async def add_note(title: str, content: str, priority: str = "medium",
                   category: str = "general", tags: str = ""):
    """Add a new note"""
    notes = load_notes()

    note = {
        "id": len(notes) + 1,
        "title": title,
        "content": content,
        "priority": priority,
        "category": category,
        "tags": [t.strip() for t in tags.split(",") if t.strip()],
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "updated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "completed": False
    }
    notes.append(note)
    save_notes(notes)

    priority_emoji = {"high": "🔴", "medium": "🟡", "low": "🟢"}
    emoji = priority_emoji.get(priority, "")
    return f"{emoji} Note #{note['id']} created: {title}"

# 2. List Notes with Filters
@server.tool(
    name="list_notes",
    description="List notes with filtering and sorting",
    input_schema={
        "type": "object",
        "properties": {
            "category": {"type": "string", "description": "Filter by category"},
            "priority": {"type": "string", "description": "Filter by priority"},
            "tag": {"type": "string", "description": "Filter by tag"},
            "status": {
                "type": "string",
                "enum": ["all", "active", "completed"],
                "description": "Filter by completion status"
            },
            "sort_by": {
                "type": "string",
                "enum": ["date", "priority", "title"],
                "description": "Sort order"
            }
        }
    }
)
async def list_notes(category: str = "", priority: str = "", tag: str = "",
                     status: str = "all", sort_by: str = "date"):
    """List notes with filters"""
    notes = load_notes()

    # Apply filters
    if category:
        notes = [n for n in notes if n.get("category") == category]
    if priority:
        notes = [n for n in notes if n.get("priority") == priority]
    if tag:
        notes = [n for n in notes if tag in n.get("tags", [])]
    if status == "active":
        notes = [n for n in notes if not n.get("completed", False)]
    elif status == "completed":
        notes = [n for n in notes if n.get("completed", False)]

    # Apply sorting
    if sort_by == "priority":
        priority_order = {"high": 0, "medium": 1, "low": 2}
        notes.sort(key=lambda n: priority_order.get(n.get("priority"), 1))
    elif sort_by == "title":
        notes.sort(key=lambda n: n["title"].lower())
    else:  # date
        notes.sort(key=lambda n: n["created_at"], reverse=True)

    if not notes:
        return "No notes found matching your criteria!"

    # Format output
    result = "Your Notes (" + str(len(notes)) + " found)\n"
    result += "=" * 40 + "\n"

    for note in notes:
        priority_emoji = {"high": "🔴", "medium": "🟡", "low": "🟢"}
        emoji = priority_emoji.get(note.get("priority"), "")
        status_mark = "✅" if note.get("completed") else "📌"
        category_text = "[" + note.get("category", "general") + "]"

        result += f"{status_mark} {emoji} #{note['id']} {note['title']} {category_text}\n"

        # Show tags if any
        if note.get("tags"):
            result += "   Tags: " + ", ".join(note['tags']) + "\n"

    return result

# 3. Read Note
@server.tool(
    name="read_note",
    description="Read a specific note by ID",
    input_schema={
        "type": "object",
        "properties": {
            "note_id": {"type": "integer", "description": "ID of note to read"}
        },
        "required": ["note_id"]
    }
)
async def read_note(note_id: int):
    """Read a specific note"""
    notes = load_notes()

    for note in notes:
        if note["id"] == note_id:
            result = f"Note #{note['id']}: {note['title']}\n"
            result += "=" * 40 + "\n\n"
            result += note['content'] + "\n\n"
            result += "Created: " + note.get('created_at', 'Unknown') + "\n"
            result += "Priority: " + note.get('priority', 'medium') + "\n"
            result += "Category: " + note.get('category', 'general') + "\n"
            result += "Status: " + ("Completed" if note.get('completed') else "Active") + "\n"

            if note.get('tags'):
                result += "Tags: " + ", ".join(note['tags']) + "\n"

            return result

    return f"Note #{note_id} not found"

# 4. Search Notes
@server.tool(
    name="search_notes",
    description="Search notes by keyword in title, content, or tags",
    input_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search query"},
            "search_in": {
                "type": "string",
                "enum": ["all", "title", "content", "tags"],
                "description": "Where to search"
            }
        },
        "required": ["query"]
    }
)
async def search_notes(query: str, search_in: str = "all"):
    """Search notes intelligently"""
    notes = load_notes()
    query = query.lower()

    matches = []
    for note in notes:
        found = False
        context = ""

        if search_in in ["all", "title"] and query in note["title"].lower():
            found = True
            context = "Title match"
        elif search_in in ["all", "content"] and query in note["content"].lower():
            found = True
            # Extract context
            content_lower = note["content"].lower()
            idx = content_lower.find(query)
            start = max(0, idx - 30)
            end = min(len(note["content"]), idx + len(query) + 30)
            snippet = note["content"][start:end]
            if start > 0:
                snippet = "..." + snippet
            if end < len(note["content"]):
                snippet = snippet + "..."
            context = snippet
        elif search_in in ["all", "tags"]:
            tag_match = False
            for tag_text in note.get("tags", []):
                if query in tag_text.lower():
                    tag_match = True
                    break
            if tag_match:
                found = True
                context = "Tag match: " + ", ".join(note['tags'])

        if found:
            matches.append({"note": note, "context": context})

    if not matches:
        return f"No notes found matching '{query}'"

    result = f"Found {len(matches)} notes matching '{query}'\n"
    result += "=" * 40 + "\n"

    for match in matches:
        note = match["note"]
        result += f"#{note['id']}: {note['title']}\n"
        result += f"   {match['context']}\n"
        result += f"   Created: {note['created_at'][:10]}\n\n"

    return result

# 5. Update Note
@server.tool(
    name="update_note",
    description="Update an existing note",
    input_schema={
        "type": "object",
        "properties": {
            "note_id": {"type": "integer", "description": "ID of note to update"},
            "title": {"type": "string", "description": "New title (optional)"},
            "content": {"type": "string", "description": "New content (optional)"},
            "priority": {"type": "string", "enum": ["high", "medium", "low"]},
            "completed": {"type": "boolean", "description": "Mark as completed"}
        },
        "required": ["note_id"]
    }
)
async def update_note(note_id: int, title: str = None, content: str = None,
                      priority: str = None, completed: bool = None):
    """Update a note"""
    notes = load_notes()

    for note in notes:
        if note["id"] == note_id:
            if title:
                note["title"] = title
            if content:
                note["content"] = content
            if priority:
                note["priority"] = priority
            if completed is not None:
                note["completed"] = completed
            note["updated_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            save_notes(notes)
            return f"Note #{note_id} updated successfully!"

    return f"Note #{note_id} not found"

# 6. Delete Note
@server.tool(
    name="delete_note",
    description="Delete a note by ID",
    input_schema={
        "type": "object",
        "properties": {
            "note_id": {"type": "integer", "description": "ID of note to delete"}
        },
        "required": ["note_id"]
    }
)
async def delete_note(note_id: int):
    """Delete a note"""
    notes = load_notes()
    original_length = len(notes)
    notes = [n for n in notes if n["id"] != note_id]

    if len(notes) < original_length:
        save_notes(notes)
        return f"Note #{note_id} deleted!"

    return f"Note #{note_id} not found"

# 7. Get Statistics
@server.tool(
    name="get_stats",
    description="Get statistics about your notes",
    input_schema={"type": "object", "properties": {}}
)
async def get_stats():
    """Get note statistics"""
    notes = load_notes()

    if not notes:
        return "No notes to analyze!"

    total = len(notes)
    completed = len([n for n in notes if n.get("completed")])
    active = total - completed

    # Category breakdown
    categories = {}
    for note in notes:
        cat = note.get("category", "general")
        categories[cat] = categories.get(cat, 0) + 1

    # Priority breakdown
    priorities = {}
    for note in notes:
        pri = note.get("priority", "medium")
        priorities[pri] = priorities.get(pri, 0) + 1

    # Most used tags
    tags_count = {}
    for note in notes:
        for tag_text in note.get("tags", []):
            tags_count[tag_text] = tags_count.get(tag_text, 0) + 1

    top_tags = sorted(tags_count.items(), key=lambda x: x[1], reverse=True)[:5]

    # Build statistics text
    completion_rate = (completed * 100 // total) if total > 0 else 0

    result = "Notes Statistics\n"

    result += f"Total Notes: {total}\n"
    result += f"Completed: {completed} ({completion_rate}%)\n"
    result += f"Active: {active} ({100 - completion_rate}%)\n\n"

    result += "Categories:\n"
    for cat, count in sorted(categories.items()):
        bar_length = count * 20 // total if total > 0 else 0
        bar = "█" * bar_length
        result += f"  {cat:15} {count:3} {bar}\n"

    result += "\nPriorities:\n"
    for pri in ["high", "medium", "low"]:
        count = priorities.get(pri, 0)
        result += f"  {pri:8} {count:3} notes\n"

    if top_tags:
        result += "\nTop Tags:\n"
        for tag_text, count in top_tags:
            result += f"  {tag_text}: {count}\n"

    return result


# Test the server

print("SMART NOTES MCP SERVER")


# Show available tools
print("\nAvailable Tools:")
for tool in server.get_tools():
    print(f"  - {tool['name']}: {tool['description']}")

# Add sample notes

print("ADDING SAMPLE NOTES")

result = await server.call_tool("add_note", {
    "title": "Complete Project Report",
    "content": "Finish the Q4 project report with all metrics and analysis",
    "priority": "high",
    "category": "work",
    "tags": "report,urgent,Q4"
})
print(result.get('result', result.get('error', 'Unknown error')))

result = await server.call_tool("add_note", {
    "title": "Buy Groceries",
    "content": "Milk, eggs, bread, vegetables, fruits",
    "priority": "medium",
    "category": "personal",
    "tags": "shopping,weekly"
})
print(result.get('result', result.get('error', 'Unknown error')))

result = await server.call_tool("add_note", {
    "title": "Learn MCP Protocol",
    "content": "Study Model Context Protocol for AI integration",
    "priority": "high",
    "category": "learning",
    "tags": "AI,programming,MCP"
})
print(result.get('result', result.get('error', 'Unknown error')))

# List all notes

print("LISTING ALL NOTES")


result = await server.call_tool("list_notes", {})
print(result.get('result', result.get('error', 'Unknown error')))

# Filter by priority

print("FILTERING HIGH PRIORITY NOTES")


result = await server.call_tool("list_notes", {"priority": "high"})
print(result.get('result', result.get('error', 'Unknown error')))

# Search notes

print("SEARCHING FOR 'AI'")


result = await server.call_tool("search_notes", {"query": "AI"})
print(result.get('result', result.get('error', 'Unknown error')))

# Read specific note

print("READING NOTE #3")


result = await server.call_tool("read_note", {"note_id": 3})
print(result.get('result', result.get('error', 'Unknown error')))

# Get statistics

print("NOTES STATISTICS")


result = await server.call_tool("get_stats", {})
print(result.get('result', result.get('error', 'Unknown error')))

# Mark as completed

print("MARKING NOTE #1 AS COMPLETED")


result = await server.call_tool("update_note", {"note_id": 1, "completed": True})
print(result.get('result', result.get('error', 'Unknown error')))

# Final statistics

print("FINAL STATISTICS")


result = await server.call_tool("get_stats", {})
print(result.get('result', result.get('error', 'Unknown error')))


print("ALL TESTS PASSED SUCCESSFULLY!")

SMART NOTES MCP SERVER

Available Tools:
  - add_note: Create a new note with title, content, priority, and category
  - list_notes: List notes with filtering and sorting
  - read_note: Read a specific note by ID
  - search_notes: Search notes by keyword in title, content, or tags
  - update_note: Update an existing note
  - delete_note: Delete a note by ID
  - get_stats: Get statistics about your notes
ADDING SAMPLE NOTES
🔴 Note #9 created: Complete Project Report
🟡 Note #10 created: Buy Groceries
🔴 Note #11 created: Learn MCP Protocol
LISTING ALL NOTES
Your Notes (11 found)
📌 🔴 #9 Complete Project Report [work]
   Tags: report, urgent, Q4
📌 🟡 #10 Buy Groceries [personal]
   Tags: shopping, weekly
📌 🔴 #11 Learn MCP Protocol [learning]
   Tags: AI, programming, MCP
📌 🔴 #6 Complete Project Report [work]
   Tags: report, urgent, Q4
📌 🟡 #7 Buy Groceries [personal]
   Tags: shopping, weekly
📌 🔴 #8 Learn MCP Protocol [learning]
   Tags: AI, programming, MCP
📌 🔴 #3 Complete Project Report [w

In [ ]:
# Final Working Version - Smart Notes MCP Server
import json
import os
from datetime import datetime
from typing import Dict, List, Any

class MCPServer:
    """MCP-like server that works in Jupyter"""

    def __init__(self, name: str):
        self.name = name
        self.tools = {}

    def tool(self, name: str, description: str, input_schema: Dict = None):
        """Decorator to register a tool"""
        def decorator(func):
            self.tools[name] = {
                "handler": func,
                "description": description,
                "input_schema": input_schema or {"type": "object", "properties": {}}
            }
            return func
        return decorator

    def get_tools(self) -> List[Dict]:
        """List all registered tools"""
        return [
            {"name": name, "description": info["description"], "inputSchema": info["input_schema"]}
            for name, info in self.tools.items()
        ]

    async def call_tool(self, name: str, arguments: Dict = None) -> Dict:
        """Call a registered tool"""
        if name not in self.tools:
            return {"error": f"Tool '{name}' not found"}

        try:
            if arguments:
                result = await self.tools[name]["handler"](**arguments)
            else:
                result = await self.tools[name]["handler"]()
            return {"success": True, "result": result}
        except Exception as e:
            import traceback
            error_details = traceback.format_exc()
            return {"error": str(e), "traceback": error_details}

# Create server
server = MCPServer("smart-notes")
NOTES_FILE = "smart_notes.json"

def load_notes():
    """Load notes from JSON file"""
    if os.path.exists(NOTES_FILE):
        try:
            with open(NOTES_FILE, 'r') as f:
                return json.load(f)
        except:
            return []
    return []

def save_notes(notes):
    """Save notes to JSON file"""
    with open(NOTES_FILE, 'w') as f:
        json.dump(notes, f, indent=2)

# 1. Add Note
@server.tool(
    name="add_note",
    description="Create a new note with title, content, priority, and category",
    input_schema={
        "type": "object",
        "properties": {
            "title": {"type": "string", "description": "Note title"},
            "content": {"type": "string", "description": "Note content"},
            "priority": {
                "type": "string",
                "enum": ["high", "medium", "low"],
                "description": "Priority level"
            },
            "category": {
                "type": "string",
                "description": "Category (work, personal, ideas, etc.)"
            },
            "tags": {
                "type": "string",
                "description": "Comma-separated tags"
            }
        },
        "required": ["title", "content"]
    }
)
async def add_note(title: str, content: str, priority: str = "medium",
                   category: str = "general", tags: str = ""):
    """Add a new note"""
    notes = load_notes()

    # Parse tags
    tag_list = []
    if tags:
        tag_list = [t.strip() for t in tags.split(",") if t.strip()]

    note = {
        "id": len(notes) + 1,
        "title": title,
        "content": content,
        "priority": priority,
        "category": category,
        "tags": tag_list,
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "updated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "completed": False
    }

    notes.append(note)
    save_notes(notes)

    priority_emoji = {"high": "🔴", "medium": "🟡", "low": "🟢"}
    emoji = priority_emoji.get(priority, "")

    return f"{emoji} Note #{note['id']} created: {title}"

# 2. List Notes (SIMPLIFIED - No f-strings issues)
@server.tool(
    name="list_notes",
    description="List notes with filtering options",
    input_schema={
        "type": "object",
        "properties": {
            "category": {"type": "string", "description": "Filter by category"},
            "priority": {"type": "string", "description": "Filter by priority"},
            "status": {
                "type": "string",
                "enum": ["all", "active", "completed"],
                "description": "Filter by completion status"
            }
        }
    }
)
async def list_notes(category: str = "", priority: str = "", status: str = "all"):
    """List notes with filters"""
    notes = load_notes()

    # Apply filters one by one
    filtered_notes = []
    for note in notes:
        # Check category filter
        if category and note.get("category", "") != category:
            continue

        # Check priority filter
        if priority and note.get("priority", "") != priority:
            continue

        # Check status filter
        if status == "active" and note.get("completed", False):
            continue
        if status == "completed" and not note.get("completed", False):
            continue

        filtered_notes.append(note)

    # Sort by ID (most recent first)
    filtered_notes.sort(key=lambda x: x["id"], reverse=True)

    if not filtered_notes:
        return "No notes found matching your criteria!"

    # Build result with simple string concatenation
    result_lines = []
    result_lines.append("Your Notes (" + str(len(filtered_notes)) + " found)")


    for note in filtered_notes:
        # Status icon
        if note.get("completed"):
            status_icon = "Completed"
        else:
            status_icon = "Not Completed"

        # Priority icon
        priority_map = {"high": "🔴", "medium": "🟡", "low": "🟢"}
        priority_icon = priority_map.get(note.get("priority", "medium"), "")

        # Build note line
        note_line = status_icon + " " + priority_icon
        note_line += " #" + str(note["id"])
        note_line += " " + note["title"]
        note_line += " [" + note.get("category", "general") + "]"

        result_lines.append(note_line)

        # Tags line
        tags = note.get("tags", [])
        if tags:
            result_lines.append("   Tags: " + ", ".join(tags))

    return "\n".join(result_lines)

# 3. Read Note
@server.tool(
    name="read_note",
    description="Read a specific note by ID",
    input_schema={
        "type": "object",
        "properties": {
            "note_id": {"type": "integer", "description": "ID of note to read"}
        },
        "required": ["note_id"]
    }
)
async def read_note(note_id: int):
    """Read a specific note"""
    notes = load_notes()

    for note in notes:
        if note["id"] == note_id:
            lines = []
            lines.append("Note #" + str(note['id']) + ": " + note['title'])

            lines.append("")
            lines.append(note['content'])
            lines.append("")
            lines.append("Created: " + note.get('created_at', 'Unknown'))
            lines.append("Priority: " + note.get('priority', 'medium'))
            lines.append("Category: " + note.get('category', 'general'))

            if note.get('completed'):
                lines.append("Status: Completed")
            else:
                lines.append("Status: Active")

            tags = note.get('tags', [])
            if tags:
                lines.append("Tags: " + ", ".join(tags))

            return "\n".join(lines)

    return "Note #" + str(note_id) + " not found"

# 4. Search Notes
@server.tool(
    name="search_notes",
    description="Search notes by keyword",
    input_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search query"}
        },
        "required": ["query"]
    }
)
async def search_notes(query: str):
    """Search notes"""
    notes = load_notes()
    query_lower = query.lower()

    matches = []
    for note in notes:
        if query_lower in note["title"].lower():
            matches.append((note, "Title match"))
        elif query_lower in note["content"].lower():
            matches.append((note, "Content match"))

    if not matches:
        return "No notes found matching '" + query + "'"

    lines = []
    lines.append("Found " + str(len(matches)) + " notes matching '" + query + "'")


    for note, match_type in matches:
        lines.append("#" + str(note['id']) + ": " + note['title'])
        lines.append("   Match: " + match_type)
        lines.append("   Created: " + note['created_at'][:10])
        lines.append("")

    return "\n".join(lines)

# 5. Update Note
@server.tool(
    name="update_note",
    description="Update an existing note",
    input_schema={
        "type": "object",
        "properties": {
            "note_id": {"type": "integer", "description": "ID of note to update"},
            "completed": {"type": "boolean", "description": "Mark as completed"}
        },
        "required": ["note_id"]
    }
)
async def update_note(note_id: int, completed: bool = False):
    """Update a note"""
    notes = load_notes()

    for note in notes:
        if note["id"] == note_id:
            note["completed"] = completed
            note["updated_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            save_notes(notes)

            if completed:
                return "Note #" + str(note_id) + " marked as completed! ✅"
            else:
                return "Note #" + str(note_id) + " updated successfully!"

    return "Note #" + str(note_id) + " not found"

# 6. Delete Note
@server.tool(
    name="delete_note",
    description="Delete a note by ID",
    input_schema={
        "type": "object",
        "properties": {
            "note_id": {"type": "integer", "description": "ID of note to delete"}
        },
        "required": ["note_id"]
    }
)
async def delete_note(note_id: int):
    """Delete a note"""
    notes = load_notes()
    original_length = len(notes)

    # Keep all notes except the one to delete
    notes = [n for n in notes if n["id"] != note_id]

    if len(notes) < original_length:
        save_notes(notes)
        return "Note #" + str(note_id) + " deleted! 🗑️"

    return "Note #" + str(note_id) + " not found"

# 7. Get Statistics
@server.tool(
    name="get_stats",
    description="Get statistics about your notes",
    input_schema={"type": "object", "properties": {}}
)
async def get_stats():
    """Get note statistics"""
    notes = load_notes()

    if not notes:
        return "No notes to analyze!"

    total = len(notes)
    completed_count = sum(1 for n in notes if n.get("completed"))
    active_count = total - completed_count

    # Category breakdown
    categories = {}
    for note in notes:
        cat = note.get("category", "general")
        categories[cat] = categories.get(cat, 0) + 1

    # Priority breakdown
    priorities = {"high": 0, "medium": 0, "low": 0}
    for note in notes:
        pri = note.get("priority", "medium")
        if pri in priorities:
            priorities[pri] += 1

    lines = []
    lines.append("Notes Statistics")

    lines.append("Total Notes: " + str(total))
    lines.append("Completed: " + str(completed_count))
    lines.append("Active: " + str(active_count))
    lines.append("")
    lines.append("Categories:")

    for cat in sorted(categories.keys()):
        count = categories[cat]
        lines.append("  " + cat + ": " + str(count) + " notes")

    lines.append("")
    lines.append("Priorities:")
    lines.append("  High: " + str(priorities["high"]) + " notes")
    lines.append("  Medium: " + str(priorities["medium"]) + " notes")
    lines.append("  Low: " + str(priorities["low"]) + " notes")

    return "\n".join(lines)


# Clear old notes for clean test
if os.path.exists(NOTES_FILE):
    os.remove(NOTES_FILE)


print("SMART NOTES MCP SERVER - WORKING VERSION")


# Show tools
print("\nAvailable Tools:")
for tool in server.get_tools():
    print("  - " + tool['name'] + ": " + tool['description'])

# Test adding notes

print("ADDING NOTES")


r = await server.call_tool("add_note", {
    "title": "Complete Project Report",
    "content": "Finish the Q4 project report with all metrics",
    "priority": "high",
    "category": "work",
    "tags": "report,urgent,Q4"
})
print(r.get('result', r.get('error', 'Unknown error')))

r = await server.call_tool("add_note", {
    "title": "Buy Groceries",
    "content": "Milk, eggs, bread, vegetables",
    "priority": "medium",
    "category": "personal",
    "tags": "shopping,weekly"
})
print(r.get('result', r.get('error', 'Unknown error')))

r = await server.call_tool("add_note", {
    "title": "Learn MCP Protocol",
    "content": "Study Model Context Protocol for AI integration",
    "priority": "high",
    "category": "learning",
    "tags": "AI,programming,MCP"
})
print(r.get('result', r.get('error', 'Unknown error')))

# Test listing

print("LISTING ALL NOTES")


r = await server.call_tool("list_notes", {})
print(r.get('result', r.get('error', 'Unknown error')))

# Test filtering

print("FILTERING HIGH PRIORITY")


r = await server.call_tool("list_notes", {"priority": "high"})
print(r.get('result', r.get('error', 'Unknown error')))

# Test search

print("SEARCHING FOR 'MCP'")


r = await server.call_tool("search_notes", {"query": "MCP"})
print(r.get('result', r.get('error', 'Unknown error')))

# Test read

print("READING NOTE #3")


r = await server.call_tool("read_note", {"note_id": 3})
print(r.get('result', r.get('error', 'Unknown error')))

# Test update

print("COMPLETING NOTE #1")


r = await server.call_tool("update_note", {"note_id": 1, "completed": True})
print(r.get('result', r.get('error', 'Unknown error')))

# Test statistics

print("NOTES STATISTICS")


r = await server.call_tool("get_stats", {})
print(r.get('result', r.get('error', 'Unknown error')))

# List again to see completion

print("UPDATED LIST (Note #1 completed)")


r = await server.call_tool("list_notes", {})
print(r.get('result', r.get('error', 'Unknown error')))


print("ALL TESTS PASSED SUCCESSFULLY!")


SMART NOTES MCP SERVER - WORKING VERSION

Available Tools:
  - add_note: Create a new note with title, content, priority, and category
  - list_notes: List notes with filtering options
  - read_note: Read a specific note by ID
  - search_notes: Search notes by keyword
  - update_note: Update an existing note
  - delete_note: Delete a note by ID
  - get_stats: Get statistics about your notes
ADDING NOTES
🔴 Note #1 created: Complete Project Report
🟡 Note #2 created: Buy Groceries
🔴 Note #3 created: Learn MCP Protocol
LISTING ALL NOTES
Your Notes (3 found)
Not Completed 🔴 #3 Learn MCP Protocol [learning]
   Tags: AI, programming, MCP
Not Completed 🟡 #2 Buy Groceries [personal]
   Tags: shopping, weekly
Not Completed 🔴 #1 Complete Project Report [work]
   Tags: report, urgent, Q4
FILTERING HIGH PRIORITY
Your Notes (2 found)
Not Completed 🔴 #3 Learn MCP Protocol [learning]
   Tags: AI, programming, MCP
Not Completed 🔴 #1 Complete Project Report [work]
   Tags: report, urgent, Q4
SEARCHING F